# Ранжирование русскоязычных фрагментов: BGE v3

Финальное решение использует `BAAI/bge-reranker-v2-m3`. Ноутбук содержит подготовку данных, фиксированное дообучение на полном train и получение `submission.csv`. История экспериментов и EDA приведены в README.

**Среда исходного запуска:** Kaggle Notebooks, Linux x86_64, две NVIDIA Tesla T4, Python 3.13.16, PyTorch 2.11.0+cu130. Сид обучения и формирования пар — 42.

Для полного запуска подключите конкурсные CSV, выберите GPU T4 ×2, включите Internet для загрузки исходной модели и пакетов и выполните все ячейки по порядку. По умолчанию выбран режим обучения с последующим инференсом.

Для вычислений на двух GPU каждая смысловая часть кода сохраняется как обычный Python-модуль через `%%writefile`; все исходные функции находятся в этом ноутбуке.


## 1. Данные, режим запуска и параметры

`MODE = "train"` запускает обе стадии обучения и инференс. Для повторного инференса установите `MODE = "predict"` и укажите `SAVED_MODEL_DIR` — полную папку `pair_stage/model` из выполненного обучения. `DATA_DIR = None` включает поиск исходных CSV в `/kaggle/input` с проверкой SHA-256; при необходимости можно указать папку явно. Каждый запуск создаёт отдельную рабочую папку и выводит её путь.


In [ ]:
import hashlib
import importlib.metadata as metadata
import json
import os
import signal
import subprocess
import sys
import tempfile
from pathlib import Path

# auto ищет готовую модель, train запускает обе стадии, predict требует готовые веса.
MODE = "train"
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")
DATA_DIR = None
SAVED_MODEL_DIR = None

if MODE not in {"auto", "train", "predict"}:
    raise ValueError("Допустимые режимы: auto, train, predict")
if sys.version_info[:2] != (3, 13):
    raise RuntimeError("Для эталонного окружения нужен Python 3.13.")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path(tempfile.mkdtemp(prefix="bge3_run_", dir=OUTPUT_ROOT))
CODE_DIR = WORK_DIR / "code"
CODE_DIR.mkdir()
CONFIG_PATH = WORK_DIR / "run_config.json"
REQUIREMENTS_PATH = WORK_DIR / "requirements.txt"

CONFIG = {
    "work_dir": str(WORK_DIR),
    "model_id": "BAAI/bge-reranker-v2-m3",
    "revision": "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e",
    "training_seed": 42,
    "full_determinism": True,
    "max_length": 512,
    "query_limit": 128,
    "title_limit": 96,
    "long_text_strategy": "query_window",
    "eval_batch_size": 8,
    "reference_preprocessor_sha256": "e3397164cd40dcc763169b0e29c13b592d35d5bbabcbfaa270b0abdcb6602110",
    "input_sha256": {
        "train_path": "2c1b26c5d3ee642f2c959b758cea49d4176ca4c0dfe92fbb7236733eff801042",
        "test_path": "4667c49296b72bf64767338c5345cb2f6c7cfbd0671364e5b728d9d5bcc2bf20",
        "sample_path": "b2cd50bc4fbaa41cacf04a74d01746c0e9ca391c4e5124822278734700cc3090",
    },
}
CONFIG["recipe"] = [
    {
        "stage": "bce",
        "learning_rate": 5e-06,
        "warmup_steps": 197,
        "horizon_steps": 3943,
        "gradient_accumulation": 8,
        "seed": 42,
        "pair_lambda": 0.0,
        "successful_steps": 873,
    },
    {
        "stage": "pair",
        "learning_rate": 1e-06,
        "warmup_steps": 25,
        "horizon_steps": 749,
        "gradient_accumulation": 4,
        "seed": 42,
        "pair_lambda": 0.2,
        "successful_steps": 499,
    },
]


def file_digest(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def is_final_checkpoint(folder):
    # Поддерживаем обычный файл и шардированный формат Hugging Face.
    required = [
        "config.json",
        "run_config.json",
        "recipe.json",
        "preprocessor.json",
        "tokenizer.json",
    ]
    if not all((folder / name).is_file() for name in required):
        return False
    try:
        saved = json.loads((folder / "run_config.json").read_text())
        recipe = json.loads((folder / "recipe.json").read_text())
        if recipe != CONFIG["recipe"] or saved.get("revision") != CONFIG["revision"]:
            return False
        if (
            file_digest(folder / "preprocessor.json")
            != CONFIG["reference_preprocessor_sha256"]
        ):
            return False
        weights = folder / "model.safetensors"
        if weights.is_file() and weights.stat().st_size > 0:
            return True
        index = json.loads((folder / "model.safetensors.index.json").read_text())
        shards = set(index["weight_map"].values())
        return bool(shards) and all(
            Path(s).name == s and (folder / s).is_file() for s in shards
        )
    except (OSError, ValueError, KeyError, TypeError):
        return False


if SAVED_MODEL_DIR is not None:
    SAVED_MODEL_DIR = Path(SAVED_MODEL_DIR)
    if not is_final_checkpoint(SAVED_MODEL_DIR):
        raise ValueError(
            "Нужна полная финальная модель с весами, рецептом и словарями BGE v3."
        )
elif MODE != "train":
    roots = (INPUT_ROOT, OUTPUT_ROOT)
    candidates = sorted(
        {
            p.parent
            for root in roots
            for p in root.rglob("recipe.json")
            if is_final_checkpoint(p.parent)
        }
    )
    if len(candidates) > 1:
        raise ValueError(
            f"Найдено несколько финальных моделей. Укажите SAVED_MODEL_DIR: {candidates}"
        )
    SAVED_MODEL_DIR = next(iter(candidates), None)

if MODE == "predict" and SAVED_MODEL_DIR is None:
    raise FileNotFoundError("Для predict подключите папку финальной модели.")
RUN_TRAINING = MODE == "train" or SAVED_MODEL_DIR is None
files = {"test_path": "test.csv", "sample_path": "sample_submission.csv"}
if RUN_TRAINING:
    files["train_path"] = "train.csv"

folders = (
    [Path(DATA_DIR)]
    if DATA_DIR
    else sorted({p.parent for p in INPUT_ROOT.rglob("sample_submission.csv")})
)
for folder in folders:
    if all(
        (folder / name).is_file()
        and file_digest(folder / name) == CONFIG["input_sha256"][key]
        for key, name in files.items()
    ):
        CONFIG.update(
            {key: str((folder / name).resolve()) for key, name in files.items()}
        )
        break
else:
    raise FileNotFoundError(
        "Подключите оригинальные CSV конкурса или укажите каталог в DATA_DIR."
    )
print("Данные:", folder)
print(
    "Режим:",
    "обучение и инференс" if RUN_TRAINING else f"инференс из {SAVED_MODEL_DIR}",
)
print("Выходная папка:", WORK_DIR)

## 2. Зафиксированные зависимости

Версии взяты из исходного запуска Kaggle. Ноутбук проверяет установленный GPU-стек и при необходимости устанавливает остальные пакеты в отдельный каталог для дочерних процессов.

In [ ]:
%%writefile {REQUIREMENTS_PATH}
# Python 3.13 / Linux. Versions captured from the BGE v3 training environment.
# The CUDA-enabled torch wheel is hosted by the official PyTorch index.
--extra-index-url https://download.pytorch.org/whl/cu130
torch==2.11.0+cu130
transformers==4.56.2
tokenizers==0.22.0
huggingface_hub==0.36.2
accelerate==1.10.1
safetensors==0.6.2
sentencepiece==0.2.1
numpy==2.1.3
pandas==2.3.3
scikit-learn==1.6.1
packaging==26.3
psutil==5.9.5
PyYAML==6.0.3
regex==2025.11.3
requests==2.34.2
tqdm==4.67.3
filelock==4.2.4
fsspec==2025.12.0
certifi==2026.10.23
charset-normalizer==3.4.9
idna==3.20
urllib3==2.8.0
hf-xet==1.7.0
typing_extensions==4.16.0
sympy==1.14.0
networkx==3.7
Jinja2==3.1.6
MarkupSafe==3.0.4
mpmath==1.3.0
scipy==1.16.3
joblib==1.6.0
threadpoolctl==3.7.0
six==1.17.0
python-dateutil==2.9.0.post0
pytz==2025.2
tzdata==2026.5
setuptools==81.0.0
triton==3.6.0

# Additional runtime dependencies of these wheels on Linux x86_64.
cloudpickle==3.1.2
cuda-bindings==13.0.3
cuda-pathfinder==1.8.4
cuda-toolkit==13.0.2
nvidia-cublas==13.1.0.3
nvidia-cuda-cupti==13.0.85
nvidia-cuda-nvrtc==13.0.88
nvidia-cuda-runtime==13.0.96
nvidia-cudnn-cu13==9.19.0.56
nvidia-cufft==12.0.0.61
nvidia-cufile==1.15.1.6
nvidia-curand==10.4.0.35
nvidia-cusolver==12.0.4.66
nvidia-cusparse==12.6.3.3
nvidia-cusparselt-cu13==0.8.0
nvidia-nccl-cu13==2.28.9
nvidia-nvjitlink==13.0.88
nvidia-nvshmem-cu13==3.4.5
nvidia-nvtx==13.0.85

In [ ]:
from packaging.requirements import Requirement

declared = []
for line in REQUIREMENTS_PATH.read_text().splitlines():
    line = line.strip()
    if line and not line.startswith(("#", "--")):
        declared.append(Requirement(line))

# Keep the original GPU stack; fail early if Kaggle changed it.
platform_names = {"torch", "triton", "cuda-bindings", "cuda-pathfinder", "cuda-toolkit"}
platform_names.update(
    r.name.lower() for r in declared if r.name.lower().startswith("nvidia-")
)
for name in sorted(platform_names):
    requirement = next(r for r in declared if r.name.lower() == name)
    actual = metadata.version(name)
    assert actual in requirement.specifier, (name, actual, str(requirement.specifier))

PRIVATE_PACKAGES = Path(tempfile.mkdtemp(prefix="bge3_final_packages_", dir="/tmp"))
to_install = []
for requirement in declared:
    if requirement.name.lower() in platform_names:
        continue
    try:
        already_matches = metadata.version(requirement.name) in requirement.specifier
    except metadata.PackageNotFoundError:
        already_matches = False
    if not already_matches:
        to_install.append(str(requirement))
if to_install:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-deps",
            "--target",
            str(PRIVATE_PACKAGES),
            *to_install,
        ],
        check=True,
    )

RUN_ENV = os.environ.copy()
RUN_ENV.update(
    PYTHONPATH=os.pathsep.join(
        [str(PRIVATE_PACKAGES), str(CODE_DIR), RUN_ENV.get("PYTHONPATH", "")]
    ),
    PYTHONHASHSEED="42",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
    USE_TF="0",
    USE_FLAX="0",
    TOKENIZERS_PARALLELISM="false",
    OMP_NUM_THREADS="2",
    MKL_NUM_THREADS="2",
    PYTHONUNBUFFERED="1",
    PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
)
print("Private packages:", to_install)

## 3. Предобработка текстов и восстановление заголовков

Сначала выполняется строго обратимый ремонт Windows-1251/UTF-8, затем очистка
Unicode, HTML и скрытых символов. Сохраняются регистр, ё, числа, пунктуация и отрицания.
Пропущенные заголовки дополняются только однозначным словарём по размеченному train.
IDF строится по уникальным тренировочным документам и нужен только для выбора
непрерывного окна длинного фрагмента. Метки test и внешние тексты не используются.

In [ ]:
%%writefile {CODE_DIR}/preprocessing.py
import hashlib, html, json, math, os, re, unicodedata
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

PREPROCESS_VERSION = "bge-m3-encoding-v2"
CP1251_CONTINUATIONS = "".join(
    bytes([i]).decode("cp1251", errors="replace") for i in range(128, 192)
)
MOJIBAKE_PAIR = re.compile("[РС][" + re.escape(CP1251_CONTINUATIONS) + "]")
TAGS = re.compile(
    r"</?(?:p|br|div|span|b|strong|i|em|u|s|del|ins|sub|sup|a|ul|ol|li|table|thead|tbody|tr|td|th|h[1-6]|blockquote|pre|code|hr)\b[^>]*>",
    re.I,
)
INVISIBLE = dict.fromkeys(map(ord, "\ufeff\u200b\u2060\u00ad"), None)
TITLE_SUFFIX = re.compile(r"\s+[-—–|]\s*Википедия\s*$", re.I)
EMPTY_TITLES = {"", "-", "—", "–", "null", "none", "nan", "n/a", "не указано"}
WORDS = re.compile(r"[a-zа-яё0-9]+", re.I)
WINDOW_STOP = set(
    "а и или но в во на за из к ко с со у о об от по до для при это как какой какая какое какие кто что где когда сколько был была было были есть является его ее их он она они мы вы чем чего кого какую каком который которая которое которые ли".split()
)


def repair_mojibake(value):
    text = str(value)
    count = len(MOJIBAKE_PAIR.findall(text))
    if count < 6 or 2 * count / max(1, len(text)) < 0.15:
        return text
    try:
        restored = text.encode("cp1251", errors="strict").decode(
            "utf-8", errors="strict"
        )
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text
    return restored if len(MOJIBAKE_PAIR.findall(restored)) < count / 4 else text


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(json_safe(value), ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    os.replace(tmp, path)


def clean_text(value, repair_encoding=True):
    # Сначала кодировка: U+00AD в mojibake может кодировать часть русской буквы.
    raw = repair_mojibake(value) if repair_encoding else str(value)
    s = unicodedata.normalize("NFC", raw).translate(INVISIBLE)
    # До неподвижной точки: корректно раскрывает &amp;quot; и вложенные сущности.
    while True:
        decoded = html.unescape(s)
        if decoded == s:
            break
        s = decoded
    s = TAGS.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()


def clean_title(value, repair_encoding=True):
    s = clean_text(value, repair_encoding=repair_encoding)
    while True:
        shorter = TITLE_SUFFIX.sub("", s).strip()
        if shorter == s:
            break
        s = shorter
    return "" if s.casefold() in EMPTY_TITLES else s


def query_key(value):
    s = (
        unicodedata.normalize("NFC", value)
        .casefold()
        .replace("ё", "е")
        .replace("\u0301", "")
    )
    return re.sub(r"\s+", " ", s).strip().rstrip("?!., ")


def lexical_terms(value):
    words = WORDS.findall(value.casefold().replace("ё", "е"))
    return {w for w in words if w not in WINDOW_STOP and (len(w) >= 3 or w.isdigit())}


def read_frame(path, labeled=False):
    df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    required = {
        "id",
        "query_id",
        "document_id",
        "article_id",
        "query",
        "title",
        "text",
        "passage_index",
        "article_passage_count",
        "article_char_count",
        "ingest_source",
    }
    assert required.issubset(df.columns), required - set(df.columns)
    assert df.id.is_unique and df.id.str.strip().ne("").all()
    assert df.query_id.str.strip().ne("").all()
    assert df.groupby("query_id")["query"].nunique().eq(1).all()
    if labeled:
        assert df.relevance.isin(["0", "1"]).all()
        df["relevance"] = df.relevance.astype("int8")
    return df


def prepare_common(df, repair_encoding=True):
    out = df.copy()
    for name in ["query", "title", "text"]:
        out[name + "_encoding_repaired"] = (
            out[name].map(repair_mojibake).ne(out[name]) if repair_encoding else False
        )
    out["query_clean"] = out["query"].map(lambda x: clean_text(x, repair_encoding))
    out["title_clean"] = out["title"].map(lambda x: clean_title(x, repair_encoding))
    out["text_clean"] = out["text"].map(lambda x: clean_text(x, repair_encoding))
    out["title_missing"] = out.title_clean.eq("")
    out["title_suffix_removed"] = out.title.map(
        lambda x: clean_text(x, repair_encoding)
    ).str.contains(TITLE_SUFFIX)
    out["text_changed"] = out.text.ne(out.text_clean)
    out["text_html_flag"] = out.text.str.contains(
        r"&(?:amp|lt|gt|quot|nbsp|#\d+);|</?(?:p|br|div|span)\b", regex=True, case=False
    )
    out["text_invisible_flag"] = out.text.str.contains(
        "[\ufeff\u200b\u2060\u00ad]", regex=True
    )
    assert out.query_clean.ne("").all(), "Пустой запрос после обработки."
    # Пустое содержание отмечаем, но кандидата не удаляем и ответ не выдумываем.
    out["text_missing"] = out.text_clean.eq("")
    return out


class TextPreprocessor:
    def __init__(self, repair_encoding=True):
        self.title_map, self.idf = {}, {}
        self.fit_rows_sha256 = None
        self.repair_encoding = repair_encoding

    def fit(self, fit):
        # Один и тот же документ в нескольких запросах не получает лишних голосов.
        valid = fit.loc[
            fit.title_clean.ne(""), ["article_id", "document_id", "title_clean"]
        ].drop_duplicates()
        doc_titles = valid.groupby(["article_id", "document_id"]).title_clean.agg(
            lambda x: sorted(set(x))
        )
        by_article = defaultdict(list)
        conflicted = set()
        for (article, _), titles in doc_titles.items():
            if len(titles) > 1:
                conflicted.add(article)
            if article and len(titles) == 1:
                by_article[article].append(titles[0])
        self.title_map = {
            article: titles[0]
            for article, titles in by_article.items()
            if article not in conflicted and len(titles) >= 2 and len(set(titles)) == 1
        }
        # IDF только по уникальным документам fit; нужен лишь для выбора длинного окна.
        documents = fit[["document_id", "text_clean"]].drop_duplicates("document_id")
        frequencies = Counter()
        for text in documents.text_clean:
            frequencies.update(lexical_terms(text))
        n = len(documents)
        self.idf = {
            term: min(6.0, math.log((n + 1) / (count + 1)) + 1)
            for term, count in sorted(frequencies.items())
        }
        self.fit_rows_sha256 = hashlib.sha256(
            "\n".join(sorted(fit.id)).encode()
        ).hexdigest()
        return self

    def transform(self, frame):
        out = frame.copy()
        recovered = out.article_id.map(self.title_map).fillna("")
        out["title_model"] = out.title_clean.mask(out.title_missing, recovered)
        out["title_recovered"] = out.title_missing & out.title_model.ne("")
        assert len(out) == len(frame) and out.id.equals(frame.id)
        return out

    def save(self, path):
        atomic_json(
            path,
            {
                "version": PREPROCESS_VERSION,
                "repair_encoding": self.repair_encoding,
                "title_map": self.title_map,
                "idf": self.idf,
                "fit_rows_sha256": self.fit_rows_sha256,
            },
        )

    @classmethod
    def load(cls, path):
        data = json.loads(Path(path).read_text())
        assert data["version"] in [PREPROCESS_VERSION, "bge-m3-common-v1"]
        obj = cls(
            repair_encoding=data.get(
                "repair_encoding", data["version"] == PREPROCESS_VERSION
            )
        )
        obj.title_map, obj.idf = data["title_map"], data["idf"]
        obj.fit_rows_sha256 = data["fit_rows_sha256"]
        return obj


def choose_window(ids, budget, query, tokenizer, preprocessor, strategy):
    if len(ids) <= budget:
        return ids, 0
    if strategy == "head":
        return ids[:budget], 0
    assert strategy == "query_window"
    terms = lexical_terms(query)
    if not terms:
        return ids[:budget], 0
    stride = max(1, budget // 2)
    starts = sorted(set(range(0, len(ids) - budget + 1, stride)) | {len(ids) - budget})
    best_start, best_score = 0, -1.0
    for start in starts:
        text = tokenizer.decode(ids[start : start + budget], skip_special_tokens=True)
        overlap = lexical_terms(text) & terms
        score = sum(preprocessor.idf.get(t, 3.0) for t in sorted(overlap))
        if score > best_score:
            best_start, best_score = start, score
    # При равенстве остаётся более раннее окно; при отсутствии совпадений — начало.
    return ids[best_start : best_start + budget], best_start

## 4. Токенизация и наборы данных

Формируется стандартная пара XLM-R: запрос и заголовок с фрагментом. Ограничение
512 токенов, запрос — до 128, заголовок — до 96. При обрезке выбирается окно текста
по пересечению слов запроса и fit-IDF. Кандидаты сохраняются все. Audit-файлы служат
контролем обработки. Числовые метаданные и идентификаторы не подаются в модель.

In [ ]:
%%writefile {CODE_DIR}/data.py
import torch
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from transformers import DataCollatorWithPadding
from preprocessing import atomic_json, choose_window


def get_tokenizer(source):
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(
        source, use_fast=True, padding_side="right", local_files_only=True
    )
    assert tokenizer.pad_token_id is not None
    return tokenizer


def encode_frame(frame, tokenizer, cfg, preprocessor, path, labeled=False):
    path = Path(path)
    audit = []
    special = tokenizer.num_special_tokens_to_add(pair=True)
    separator = tokenizer.encode("\n\n", add_special_tokens=False)
    tmp = path.with_suffix(".tmp")
    with tmp.open("w", encoding="utf-8") as output:
        for row in frame.itertuples(index=False):
            query = tokenizer.encode(row.query_clean, add_special_tokens=False)
            title = tokenizer.encode(row.title_model, add_special_tokens=False)
            body = tokenizer.encode(row.text_clean, add_special_tokens=False)
            q = query[: cfg["query_limit"]]
            title_head = title[: cfg["title_limit"]] + separator if title else []
            budget = cfg["max_length"] - special - len(q) - len(title_head)
            assert budget >= 64, "Недостаточно места для текста фрагмента."
            visible_query = tokenizer.decode(q, skip_special_tokens=True)
            selected, start = choose_window(
                body,
                budget,
                visible_query,
                tokenizer,
                preprocessor,
                cfg["long_text_strategy"],
            )
            # Стандартная пара XLM-R: <s> query </s></s> title+passage </s>.
            # Не используем чат-шаблон и yes/no-токены Qwen.
            ids = tokenizer.build_inputs_with_special_tokens(q, title_head + selected)
            assert len(ids) <= cfg["max_length"]
            item = {"input_ids": ids, "attention_mask": [1] * len(ids)}
            if labeled:
                item["labels"] = int(row.relevance)
            output.write(json.dumps(item, separators=(",", ":")) + "\n")
            audit.append(
                {
                    "id": row.id,
                    "query_id": row.query_id,
                    "tokens": len(ids),
                    "body_tokens": len(body),
                    "body_budget": budget,
                    "truncated": len(body) > budget,
                    "window_start": start,
                    "query_truncated": len(query) > cfg["query_limit"],
                    "title_truncated": len(title) > cfg["title_limit"],
                    "title_missing": bool(row.title_missing),
                    "title_recovered": bool(row.title_recovered),
                }
            )
            audit[-1]["text_encoding_repaired"] = bool(row.text_encoding_repaired)
            if len(audit) % 5000 == 0:
                print(path.name, len(audit), "/", len(frame), flush=True)
    os.replace(tmp, path)
    audit = pd.DataFrame(audit)
    audit.to_csv(path.with_name(path.stem + "_audit.csv"), index=False)
    summary = {
        "rows": len(frame),
        "p50_tokens": float(audit.tokens.median()),
        "p95_tokens": float(audit.tokens.quantile(0.95)),
        **{
            k: int(audit[k].sum())
            for k in [
                "truncated",
                "query_truncated",
                "title_truncated",
                "title_missing",
                "title_recovered",
            ]
        },
        "non_head_windows": int(audit.window_start.gt(0).sum()),
    }
    atomic_json(path.with_name(path.stem + "_summary.json"), summary)
    print(path.name, json.dumps(summary, ensure_ascii=False), flush=True)
    return summary


class TokenDataset(Dataset):
    def __init__(self, path, longest=None):
        with open(path, encoding="utf-8") as stream:
            self.rows = [json.loads(line) for line in stream]
        if longest:
            self.rows = sorted(
                self.rows, key=lambda x: len(x["input_ids"]), reverse=True
            )[:longest]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


class PairDataset(Dataset):
    def __init__(self, tokens, pairs):
        self.tokens, self.pairs = tokens, pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        pair = self.pairs[index]
        return {
            "left": self.tokens[pair["positive_index"]],
            "right": self.tokens[pair["negative_index"]],
            "pair_weight": pair["weight"],
        }


class TrainingCollator:
    def __init__(self, tokenizer):
        self.padding = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

    def __call__(self, features):
        if "left" not in features[0]:
            return self.padding(features)
        flattened = []
        for item in features:
            assert item["left"]["labels"] == 1 and item["right"]["labels"] == 0
            flattened.extend([item["left"], item["right"]])
        batch = self.padding(flattened)
        batch["pair_weights"] = torch.tensor(
            [x["pair_weight"] for x in features], dtype=torch.float32
        )
        return batch


def make_submission(meta, scores, sample):
    scores = np.asarray(scores).reshape(-1)
    assert len(scores) == len(meta) and np.isfinite(scores).all()
    assert meta.id.is_unique and sample.id.is_unique
    assert set(meta.id) == set(sample.id), "Набор id не совпадает с sample_submission."
    result = pd.DataFrame({"id": sample.id.astype(str)})
    result["prediction"] = result.id.map(pd.Series(scores, index=meta.id))
    assert result.prediction.notna().all() and len(result) == len(sample)
    assert result.columns.tolist() == ["id", "prediction"]
    return result

## 5. Обучающие пары и расписание скорости обучения

После BCE все тренировочные кандидаты оцениваются моделью. Для каждого
релевантного фрагмента выбираются сложный нерелевантный из той же статьи,
если он есть, иначе самый высоко оценённый отрицательный, и ещё один случайный.
Веса учитывают ΔNDCG@10 с нижней границей 0.05 и число положительных ответов.
Пары выбираются только внутри исходного запроса; seed=42. Они фиксируются на
всю парную стадию — это исходный алгоритм v3, без групповых изменений v4.

In [ ]:
%%writefile {CODE_DIR}/pairs.py
import math
import numpy as np


def mine_training_pairs(meta, scores, seed=42):
    frame = meta.reset_index(drop=True).copy()
    frame["row_index"] = np.arange(len(frame))
    frame["score"] = np.asarray(scores).reshape(-1)
    assert np.isfinite(frame.score).all()
    rng = np.random.default_rng(seed)
    result = []
    for query_id, group in frame.groupby("query_id", sort=False):
        ordered = group.sort_values(["score", "id"], ascending=[False, True])
        positives = ordered.loc[ordered.relevance.eq(1)]
        negatives = ordered.loc[ordered.relevance.eq(0)]
        if positives.empty or negatives.empty:
            continue
        rank = {rid: i for i, rid in enumerate(ordered.id)}
        discounts = 1 / np.log2(np.arange(min(10, len(group))) + 2)
        idcg = float(discounts[: min(len(positives), 10)].sum())

        def discount(index):
            return 1 / math.log2(index + 2) if index < 10 else 0.0

        for positive in positives.itertuples():
            same_article = negatives.loc[negatives.article_id.eq(positive.article_id)]
            hard = same_article.iloc[0] if len(same_article) else negatives.iloc[0]
            choices = [
                (hard, "same_article_hard" if len(same_article) else "global_hard")
            ]
            others = negatives.loc[negatives.id.ne(hard.id)]
            if len(others):
                choices.append((others.iloc[int(rng.integers(len(others)))], "random"))
            for negative, kind in choices:
                delta = (
                    abs(discount(rank[positive.id]) - discount(rank[negative.id]))
                    / idcg
                )
                result.append(
                    {
                        "query_id": query_id,
                        "positive_id": positive.id,
                        "negative_id": negative.id,
                        "positive_index": int(positive.row_index),
                        "negative_index": int(negative.row_index),
                        "kind": kind,
                        "weight": max(0.05, delta) / len(positives),
                    }
                )
    assert result, (
        "Не найдены обучающие запросы с положительными и отрицательными кандидатами."
    )
    average = sum(p["weight"] for p in result) / len(result)
    for pair in result:
        pair["weight"] /= average
    return result


def schedule_factor(step, warmup_steps, horizon_steps):
    assert horizon_steps > warmup_steps >= 0
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    return max(0.0, (horizon_steps - step) / (horizon_steps - warmup_steps))

## 6. Модель, GPU и сиды

Все параметры модели хранятся в FP32. Прямой проход использует FP16 autocast.
Каждый из двух GPU-процессов получает seed=42 и детерминированный режим.
При инференсе строки распределяются без дублирования, а оценки возвращаются
в исходном порядке. Скоры — исходные logits; sigmoid для ранжирования не нужен.

In [ ]:
%%writefile {CODE_DIR}/modeling.py
import os
from datetime import timedelta
import torch
import torch.distributed as dist
from torch.utils.data import DataLoader, Subset
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    set_seed,
)
from transformers.trainer_utils import enable_full_determinism


def main_rank():
    return not dist.is_initialized() or dist.get_rank() == 0


def sync():
    if dist.is_initialized():
        dist.barrier(device_ids=[torch.cuda.current_device()])


def initialize_distributed(cfg, seed):
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
    if cfg["full_determinism"]:
        enable_full_determinism(seed)
    else:
        set_seed(seed)
    if int(os.environ.get("WORLD_SIZE", 1)) > 1 and not dist.is_initialized():
        dist.init_process_group(
            backend="nccl",
            device_id=torch.device("cuda", local_rank),
            timeout=timedelta(hours=3),
        )


def load_model(cfg, source=None, trainable=True):
    device = int(os.environ.get("LOCAL_RANK", 0))
    device = (
        torch.device("cuda", device)
        if torch.cuda.is_available()
        else torch.device("cpu")
    )
    if device.type == "cuda":
        torch.cuda.set_device(device)
    source = str(source or cfg["base_local_dir"])
    # Параметры FP32 + FP16 autocast. Не вызываем model.half() при обучении.
    model, loading = AutoModelForSequenceClassification.from_pretrained(
        source,
        output_loading_info=True,
        torch_dtype=torch.float32,
        attn_implementation="sdpa",
        local_files_only=True,
    )
    if (
        loading["missing_keys"]
        or loading["mismatched_keys"]
        or loading.get("error_msgs")
    ):
        raise ValueError(f"Checkpoint загрузился не полностью: {loading}")
    assert model.config.model_type == "xlm-roberta" and model.config.num_labels == 1
    model.config.use_cache = False
    model.requires_grad_(trainable)
    model.to(device)
    return model


def predict_scores(model, dataset, tokenizer, batch_size):
    # Индексы rank::world не дублируют последний пакет. На выходе исходный порядок.
    rank = dist.get_rank() if dist.is_initialized() else 0
    world = dist.get_world_size() if dist.is_initialized() else 1
    indices = list(range(rank, len(dataset), world))
    loader = DataLoader(
        Subset(dataset, indices),
        batch_size=batch_size,
        collate_fn=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8),
        num_workers=0,
    )
    device = next(model.parameters()).device
    scores = torch.zeros(len(dataset), device=device, dtype=torch.float32)
    counts = torch.zeros(len(dataset), device=device, dtype=torch.int32)
    previous_mode = model.training
    model.eval()
    cursor = 0
    with torch.inference_mode():
        for batch in loader:
            batch.pop("labels", None)
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast(
                device.type, dtype=torch.float16, enabled=device.type == "cuda"
            ):
                logits = model(**batch, return_dict=True).logits.reshape(-1).float()
            if not torch.isfinite(logits).all():
                raise FloatingPointError("NaN/Inf при инференсе.")
            rows = torch.tensor(indices[cursor : cursor + len(logits)], device=device)
            scores[rows] = logits
            counts[rows] = 1
            cursor += len(logits)
    if dist.is_initialized():
        dist.all_reduce(scores, op=dist.ReduceOp.SUM)
        dist.all_reduce(counts, op=dist.ReduceOp.SUM)
    assert counts.eq(1).all(), "Пропущенные или продублированные строки инференса."
    model.train(previous_mode)
    return scores.cpu().numpy()

## 7. Функция потерь и оптимизатор

Первая стадия минимизирует BCE. Вторая — BCE + 0.2 × weighted softplus(s_neg − s_pos).
AdamW хранит состояния в FP32, foreach=False уменьшает пиковую память. Весовой
штраф 0.01 применяется к выбранным Trainer параметрам, исключая bias/LayerNorm.

In [ ]:
%%writefile {CODE_DIR}/losses.py
import torch
import torch.nn.functional as F
from transformers import Trainer
from pairs import schedule_factor


class BGETrainer(Trainer):
    def __init__(self, *args, profile, **kwargs):
        self.profile = profile
        super().__init__(*args, **kwargs)
        self.model_accepts_loss_kwargs = False

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        labels = inputs["labels"].float().reshape(-1)
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            return_dict=True,
        )
        logits = outputs.logits.float().reshape(-1)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        if "pair_weights" in inputs:
            pair_logits = logits.reshape(-1, 2)
            pair_loss = F.softplus(pair_logits[:, 1] - pair_logits[:, 0])
            loss = (
                loss
                + self.profile.get("pair_lambda", 0.0)
                * (pair_loss * inputs["pair_weights"].float()).mean()
            )
        if not torch.isfinite(loss):
            raise FloatingPointError("BCE содержит NaN/Inf; остановка обучения.")
        return (loss, {"logits": logits[:, None]}) if return_outputs else loss

    def create_optimizer(self):
        if self.optimizer is None:
            decay_names = set(self.get_decay_parameter_names(self.model))
            groups = [
                {
                    "params": [
                        p
                        for n, p in self.model.named_parameters()
                        if p.requires_grad and n in decay_names
                    ],
                    "weight_decay": self.args.weight_decay,
                },
                {
                    "params": [
                        p
                        for n, p in self.model.named_parameters()
                        if p.requires_grad and n not in decay_names
                    ],
                    "weight_decay": 0.0,
                },
            ]
            # foreach=False уменьшает пиковую память AdamW на T4.
            self.optimizer = torch.optim.AdamW(
                groups,
                lr=self.args.learning_rate,
                betas=(self.args.adam_beta1, self.args.adam_beta2),
                eps=self.args.adam_epsilon,
                foreach=False,
            )
        return self.optimizer

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        labels = inputs.get("labels")
        with torch.no_grad(), self.compute_loss_context_manager():
            logits = (
                model(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    return_dict=True,
                )
                .logits.float()
                .reshape(-1)
            )
            loss = (
                None
                if labels is None
                else F.binary_cross_entropy_with_logits(
                    logits, labels.float().reshape(-1)
                )
            )
        if not torch.isfinite(logits).all():
            raise FloatingPointError("Предсказания содержат NaN/Inf.")
        return (
            None if loss is None else loss.detach(),
            None if prediction_loss_only else logits[:, None].detach(),
            None if prediction_loss_only or labels is None else labels.detach(),
        )

    def create_scheduler(self, num_training_steps, optimizer=None):
        if self.lr_scheduler is None:
            self.lr_scheduler = torch.optim.lr_scheduler.LambdaLR(
                optimizer or self.optimizer,
                lambda step: schedule_factor(
                    step, self.profile["warmup_steps"], self.profile["horizon_steps"]
                ),
            )
        return self.lr_scheduler

## 8. Обучение по фиксированному рецепту

Сохраняются два исходных расписания: BCE warmup=197, horizon=3943; pair warmup=25,
horizon=749. Эти горизонты не равны числу реально выполняемых шагов, и заменять
их на 873/499 нельзя. Пакет BCE: 1 документ × 2 GPU × 8 накоплений = 16 документов;
парная стадия: 1 пара × 2 документа × 2 GPU × 4 накопления = 16 документов.
AMP начинается со scale=1024, clipping=1.0, checkpointing use_reentrant=False.
Валидации и выбора лучшего шага здесь нет. Считаются успешные обновления optimizer.

In [ ]:
%%writefile {CODE_DIR}/training.py
"""Two fixed training stages. No validation, hyperparameter search or EDA."""

import gc
import hashlib
import json
import math
import shutil
import time
from pathlib import Path

import pandas as pd
import torch
import torch.distributed as dist
from transformers import TrainerCallback, TrainingArguments

from preprocessing import atomic_json
from data import TokenDataset, PairDataset, TrainingCollator, get_tokenizer
from pairs import mine_training_pairs
from modeling import main_rank, sync, load_model, predict_scores
from losses import BGETrainer


class SuccessfulStepCounter(TrainerCallback):
    """Use successful AMP optimizer updates, not attempted updates, as the limit."""

    def __init__(self, target, folder):
        self.target = target
        self.folder = Path(folder)
        self.successful = 0
        self.skipped = 0
        self.rows = []
        self.trainer = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.started = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        skipped = bool(self.trainer.accelerator.optimizer_step_was_skipped)
        self.skipped += int(skipped)
        self.successful += int(not skipped)
        scaler = self.trainer.accelerator.scaler
        self.rows.append(
            {
                "attempt": state.global_step,
                "successful_steps": self.successful,
                "skipped_steps": self.skipped,
                "loss_scale": float(scaler.get_scale()),
                "elapsed_seconds": time.time() - self.started,
            }
        )
        stop = torch.tensor(int(self.successful >= self.target), device="cuda")
        if dist.is_initialized():
            dist.broadcast(stop, src=0)
        if stop.item():
            control.should_training_stop = True
        if main_rank() and (state.global_step % 50 == 0 or stop.item()):
            pd.DataFrame(self.rows).to_csv(
                self.folder / "optimizer_steps.csv", index=False
            )
            print(
                f"Successful steps: {self.successful}/{self.target}; "
                f"skipped: {self.skipped}; minutes: {(time.time() - self.started) / 60:.1f}",
                flush=True,
            )
        return control


def train_stage(cfg, profile, source, folder):
    folder = Path(folder)
    signature = hashlib.sha256(
        json.dumps(
            {
                "profile": profile,
                "inputs": cfg["input_sha256"],
                "preprocessor": cfg["preprocessor_sha256"],
                "code": cfg["code_sha256"],
            },
            sort_keys=True,
        ).encode()
    ).hexdigest()

    # Rank 0 decides whether a stage exists; all ranks receive the same decision.
    # This prevents a race between checking a marker and creating it.
    status = 0
    if main_rank():
        folder.mkdir(parents=True, exist_ok=True)
        if (folder / "done.json").exists():
            previous = json.loads((folder / "done.json").read_text())
            status = (
                1
                if previous["signature"] == signature
                and any((folder / "model").glob("*.safetensors"))
                else 3
            )
        elif (folder / "started.json").exists():
            status = 2
    flag = torch.tensor(status, device="cuda")
    if dist.is_initialized():
        dist.broadcast(flag, src=0)
    status = int(flag.item())
    if status == 1:
        return folder / "model"
    if status in [2, 3]:
        raise RuntimeError(
            f"Unfinished or different stage in {folder}. "
            "Keep its files and use a new empty work directory for a new run."
        )
    if main_rank():
        atomic_json(
            folder / "started.json", {"signature": signature, "profile": profile}
        )
    sync()

    work = Path(cfg["work_dir"])
    tokenizer = get_tokenizer(work / "tokenizer")
    tokens = TokenDataset(work / "train_tokens.jsonl")
    meta = pd.read_csv(work / "train_meta.csv", dtype={"id": str, "query_id": str})
    meta["relevance"] = meta.relevance.astype(int)
    assert len(tokens) == len(meta) == 31530
    model = load_model(cfg, source=source, trainable=True)

    if profile["stage"] == "pair":
        scores = predict_scores(model, tokens, tokenizer, cfg["eval_batch_size"])
        pairs = mine_training_pairs(meta, scores, seed=profile["seed"])
        dataset = PairDataset(tokens, pairs)
        if main_rank():
            pd.DataFrame(pairs).to_csv(folder / "training_pairs.csv", index=False)
    else:
        dataset = tokens

    # The reference scheduler horizon is deliberately longer than this stage.
    # Changing it to the number of trained steps would change the v3 recipe.
    args = TrainingArguments(
        output_dir=str(folder / "trainer_output"),
        max_steps=math.ceil(profile["successful_steps"] * 1.15) + 20,
        num_train_epochs=100,
        learning_rate=profile["learning_rate"],
        weight_decay=0.01,
        max_grad_norm=1.0,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=profile["gradient_accumulation"],
        per_device_eval_batch_size=cfg["eval_batch_size"],
        fp16=True,
        bf16=False,
        optim="adamw_torch",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="no",
        save_strategy="no",
        save_only_model=True,
        save_total_limit=2,
        save_safetensors=True,
        load_best_model_at_end=False,
        logging_steps=25,
        report_to=[],
        disable_tqdm=True,
        seed=42,
        data_seed=42,
        full_determinism=True,
        ddp_find_unused_parameters=False,
        ddp_timeout=10800,
        dataloader_num_workers=0,
        remove_unused_columns=False,
        label_names=["labels"],
        eval_accumulation_steps=8,
    )
    counter = SuccessfulStepCounter(profile["successful_steps"], folder)
    trainer = BGETrainer(
        model=model,
        args=args,
        train_dataset=dataset,
        profile=profile,
        processing_class=tokenizer,
        data_collator=TrainingCollator(tokenizer),
        callbacks=[counter],
    )
    trainer.accelerator.scaler = torch.amp.GradScaler(
        "cuda", init_scale=1024.0, growth_interval=2000
    )
    counter.trainer = trainer
    result = trainer.train()
    assert counter.successful == profile["successful_steps"], (
        "The fixed training recipe did not complete."
    )

    target = folder / "model"
    trainer.save_model(str(target))
    if main_rank():
        tokenizer.save_pretrained(target)
        shutil.copy2(work / "preprocessor.json", target / "preprocessor.json")
        atomic_json(target / "run_config.json", cfg)
        atomic_json(
            target / "recipe.json",
            cfg["recipe"][: 1 if profile["stage"] == "bce" else 2],
        )
        shutil.copy2(work / "requirements.txt", target / "requirements.txt")
        shutil.copy2(work / "environment.json", target / "environment.json")
        code = target / "code"
        code.mkdir(exist_ok=True)
        for path in Path(__file__).parent.glob("*.py"):
            shutil.copy2(path, code / path.name)
        atomic_json(
            folder / "done.json",
            {
                "signature": signature,
                "complete": True,
                "successful_steps": counter.successful,
                "skipped_steps": counter.skipped,
                "runtime_seconds": result.metrics["train_runtime"],
            },
        )
        print("SAVED:", target, flush=True)
    sync()
    counter.trainer = None
    trainer.accelerator.free_memory()
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    return target

### Проверка данных и инференс

В этом модуле находятся функции проверки окружения, подготовки train и формирования итогового файла. Входные файлы сверяются с исходными данными соревнования.


In [ ]:
%%writefile {CODE_DIR}/pipeline.py
"""Final preprocessing, fixed training and inference entry points."""

import argparse
import hashlib
import importlib.metadata as metadata
import json
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
from huggingface_hub import snapshot_download
from packaging.requirements import Requirement

from preprocessing import atomic_json, read_frame, prepare_common, TextPreprocessor
from data import get_tokenizer, encode_frame, TokenDataset, make_submission
from modeling import initialize_distributed, main_rank, sync, load_model, predict_scores
from training import train_stage


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(2**20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def check_environment(cfg):
    requirements = Path(cfg["work_dir"]) / "requirements.txt"
    declared = []
    for line in requirements.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith(("#", "--")):
            requirement = Requirement(line)
            if requirement.marker is None or requirement.marker.evaluate():
                actual = metadata.version(requirement.name)
                assert actual in requirement.specifier, (
                    requirement.name,
                    actual,
                    str(requirement.specifier),
                )
                declared.append(requirement.name)
    # Check active dependencies of the declared libraries; unrelated Kaggle
    # applications are not part of the training environment.
    for name in declared:
        for text in metadata.requires(name) or []:
            dependency = Requirement(text)
            if dependency.marker is not None and not dependency.marker.evaluate(
                {"extra": ""}
            ):
                continue
            actual = metadata.version(dependency.name)
            assert actual in dependency.specifier, (
                name,
                dependency.name,
                actual,
                str(dependency.specifier),
            )
    assert platform.system() == "Linux"
    cards = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    if cfg["run_training"] and (
        len(cards) != 2 or any("T4" not in card for card in cards)
    ):
        raise RuntimeError("Для обучения исходного рецепта выберите две Tesla T4.")
    environment = {
        "system": platform.platform(),
        "python": sys.version,
        "cuda": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "gpu": cards,
        "packages": {name: metadata.version(name) for name in declared},
        "seed": 42,
        "full_determinism": True,
    }
    atomic_json(Path(cfg["work_dir"]) / "environment.json", environment)
    print(json.dumps(environment, ensure_ascii=False, indent=2), flush=True)
    print("Declared requirements and active dependencies: OK", flush=True)


def prepare_training(cfg):
    work = Path(cfg["work_dir"])
    for key, expected in cfg["input_sha256"].items():
        assert sha256(cfg[key]) == expected, (
            f"Input differs from the competition data: {key}"
        )
    base = snapshot_download(
        cfg["model_id"],
        revision=cfg["revision"],
        cache_dir="/tmp/bge3_final_hf_cache",
        allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"],
    )
    cfg["base_local_dir"] = base
    tokenizer = get_tokenizer(base)
    tokenizer.save_pretrained(work / "tokenizer")

    # Final model: the dictionaries use all training documents, never test.
    train = prepare_common(read_frame(cfg["train_path"], labeled=True))
    assert len(train) == 31530 and train.query_id.nunique() == 3042
    preprocessor = TextPreprocessor().fit(train)
    preprocessor.save(work / "preprocessor.json")
    cfg["preprocessor_sha256"] = sha256(work / "preprocessor.json")
    assert cfg["preprocessor_sha256"] == cfg["reference_preprocessor_sha256"], (
        "Preprocessing differs from the original v3 run; stop before training."
    )
    train = preprocessor.transform(train)
    encode_frame(
        train, tokenizer, cfg, preprocessor, work / "train_tokens.jsonl", labeled=True
    )
    train[["id", "query_id", "relevance", "article_id", "document_id"]].to_csv(
        work / "train_meta.csv", index=False
    )
    audit = json.loads((work / "train_tokens_summary.json").read_text())
    assert audit["truncated"] == 825 and audit["non_head_windows"] == 54
    atomic_json(work / "run_config.json", cfg)
    print("Training preprocessing: OK", flush=True)


def inference(cfg, model_folder):
    work = Path(cfg["work_dir"])
    model_folder = Path(model_folder)
    assert any(model_folder.glob("*.safetensors")), model_folder
    saved = json.loads((model_folder / "run_config.json").read_text())
    prep = TextPreprocessor.load(model_folder / "preprocessor.json")
    tokenizer = get_tokenizer(model_folder)
    token_path = work / "test_tokens.jsonl"
    if main_rank():
        for key in ["test_path", "sample_path"]:
            assert sha256(cfg[key]) == cfg["input_sha256"][key], key
        test = prep.transform(
            prepare_common(
                read_frame(cfg["test_path"]), repair_encoding=prep.repair_encoding
            )
        )
        assert len(test) == 13507
        encode_frame(test, tokenizer, saved, prep, token_path)
        test[["id", "query_id"]].to_csv(work / "test_meta.csv", index=False)
    sync()
    model = load_model(saved, source=model_folder, trainable=False)
    scores = predict_scores(
        model, TokenDataset(token_path), tokenizer, cfg["eval_batch_size"]
    )
    if main_rank():
        test = pd.read_csv(work / "test_meta.csv", dtype=str, keep_default_na=False)
        sample = pd.read_csv(
            cfg["sample_path"], dtype={"id": str}, keep_default_na=False
        )
        submission = make_submission(test, scores, sample)
        target = work / "submission.csv"
        submission.to_csv(target, index=False)
        manifest = {
            "model": str(model_folder),
            "submission_sha256": sha256(target),
            "rows": len(submission),
            "seed": 42,
            "recipe": json.loads((model_folder / "recipe.json").read_text()),
            "loaded_saved_model": True,
            "all_predictions_finite": bool(np.isfinite(scores).all()),
            "weights_sha256": {
                path.name: sha256(path) for path in model_folder.glob("*.safetensors")
            },
        }
        atomic_json(work / "run_manifest.json", manifest)
        print("SUBMISSION:", target, flush=True)
        print(json.dumps(manifest, ensure_ascii=False, indent=2), flush=True)
    sync()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument(
        "--action",
        required=True,
        choices=["check", "prepare", "bce", "pair", "predict"],
    )
    parser.add_argument("--model")
    args = parser.parse_args()
    cfg = json.loads(Path(args.config).read_text())
    work = Path(cfg["work_dir"])
    work.mkdir(parents=True, exist_ok=True)
    if args.action == "check":
        check_environment(cfg)
        return
    if args.action == "prepare":
        prepare_training(cfg)
        return
    initialize_distributed(cfg, 42)
    try:
        if args.action == "bce":
            train_stage(
                cfg, cfg["recipe"][0], cfg["base_local_dir"], work / "bce_stage"
            )
        elif args.action == "pair":
            anchor = work / "bce_stage" / "model"
            assert (work / "bce_stage" / "done.json").exists(), "Finish BCE first."
            train_stage(cfg, cfg["recipe"][1], anchor, work / "pair_stage")
        elif args.action == "predict":
            inference(cfg, args.model or work / "pair_stage" / "model")
    finally:
        if dist.is_initialized():
            dist.destroy_process_group()


if __name__ == "__main__":
    main()

## 9. Проверка окружения и подготовка данных

Здесь начинается выполнение конвейера. Проверяются зависимости и GPU; затем для полного обучения подготавливаются train и токенизация.


In [ ]:
CONFIG["run_training"] = RUN_TRAINING
CONFIG["code_sha256"] = hashlib.sha256(
    b"".join(
        path.name.encode() + b"\0" + path.read_bytes()
        for path in sorted(CODE_DIR.glob("*.py"))
    )
).hexdigest()
CONFIG_PATH.write_text(json.dumps(CONFIG, ensure_ascii=False, indent=2))


def launch(action, model=None):
    script = CODE_DIR / "pipeline.py"
    single_process = action in ["check", "prepare"]
    if action == "predict":
        environment = json.loads((WORK_DIR / "environment.json").read_text())
        single_process = len(environment["gpu"]) < 2
    if single_process:
        command = [sys.executable, str(script)]
    else:
        command = [
            sys.executable,
            "-m",
            "torch.distributed.run",
            "--standalone",
            "--nnodes=1",
            "--nproc_per_node=2",
            str(script),
        ]
    command += ["--config", str(CONFIG_PATH), "--action", action]
    if model is not None:
        command += ["--model", str(model)]
    log_path = WORK_DIR / (action + ".log")
    with log_path.open("a", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            env=RUN_ENV,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            start_new_session=True,
        )
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            status = process.wait()
        except KeyboardInterrupt:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            raise
    if status:
        raise RuntimeError(
            f"{action}: код возврата {status}. Причина выше; полный журнал: {log_path}"
        )


launch("check")
if RUN_TRAINING:
    launch("prepare")
    CONFIG = json.loads(CONFIG_PATH.read_text())

## 10. Первый этап: BCE

Модель обучается 873 успешных шага на парах «запрос — фрагмент». Завершение этапа необходимо для запуска следующей ячейки.


In [ ]:
if RUN_TRAINING:
    launch("bce")

## 11. Второй этап: попарное дообучение

Модель после BCE используется для отбора обучающих пар. Затем выполняется 499 успешных обновлений. Итоговая модель сохраняется в `pair_stage/model` внутри рабочей папки.


In [ ]:
if RUN_TRAINING:
    launch("pair")

## 12. Инференс и сохранение результата

Загружается сохранённая модель, затем каждому кандидату тестовой выборки присваивается скор. Пути к `submission.csv`, папке модели и журналу прогона выводятся после выполнения ячейки.


In [ ]:
MODEL_DIR = WORK_DIR / "pair_stage" / "model" if RUN_TRAINING else Path(SAVED_MODEL_DIR)
launch("predict", model=MODEL_DIR)
print("Файл для Kaggle:", WORK_DIR / "submission.csv")
print("Сохранённая модель:", MODEL_DIR)
print("Проверка прогона:", WORK_DIR / "run_manifest.json")